In [16]:
# 필요한 경우 한 번만 실행
# !pip install pymupdf pymupdf4llm python-dotenv

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
import os
os.environ['PYTHONUTF8'] = '1'
os.environ['PYMUPDF4LLM_USE_LAYOUT'] = '0'
import re
import unicodedata
import pymupdf
from glob import glob
from pathlib import Path

In [19]:
import re, unicodedata

def _case_like(src: str, target: str) -> str:
    if src.isupper():
        return target.upper()
    if src[:1].isupper():
        return target[:1].upper() + target[1:]
    return target

def _sub_word(text: str, pattern: str, target: str) -> str:
    return re.sub(pattern, lambda m: _case_like(m.group(0), target), text, flags=re.IGNORECASE)

def fix_text_quality(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\ufb00', 'ff').replace('\ufb01', 'fi').replace('\ufb02', 'fl').replace('\ufb03', 'ffi').replace('\ufb04', 'ffl')
    # Markdown/OCR artifacts where ff/fl/fi is often lost as `, _, ^, W, g, aa, etc.
    patterns = [
        (r'\bo[`_wg ]icially\b', 'officially'),
        (r'\bo[`_wg ]icials\b', 'officials'),
        (r'\bo[`_wg ]icial\b', 'official'),
        (r'\bo[`_wg ]icers\b', 'officers'),
        (r'\bo[`_wg ]icer\b', 'officer'),
        (r'\bo[`_wg ]ice\b', 'office'),
        (r'\bo[`_wg ]ences\b', 'offences'),
        (r'\bo[`_wg ]ence\b', 'offence'),
        (r'\bo[`_wg ]ers\b', 'offers'),
        (r'\bo[`_wg ]er\b', 'offer'),
        (r'\be[`_ ]ective\b', 'effective'),
        (r'\be[`_ ]ectively\b', 'effectively'),
        (r'\be[`_ ]ects\b', 'effects'),
        (r'\be[`_ ]ect\b', 'effect'),
        (r'\be[`_ ]iciency\b', 'efficiency'),
        (r'\be[`_ ]orts\b', 'efforts'),
        (r'\be[`_ ]ort\b', 'effort'),
        (r'\ba[`_ ]ected\b', 'affected'),
        (r'\ba[`_ ]ecting\b', 'affecting'),
        (r'\ba[`_ ]ects\b', 'affects'),
        (r'\ba[`_ ]ect\b', 'affect'),
        (r'\ba[`_ ]ix\b', 'affix'),
        (r'\bdi[`_ ]erentiates\b', 'differentiates'),
        (r'\bdi[`_ ]erentiate\b', 'differentiate'),
        (r'\bdi[`_ ]erential\b', 'differential'),
        (r'\bdi[`_ ]erences\b', 'differences'),
        (r'\bdi[`_ ]erence\b', 'difference'),
        (r'\bdi[`_ ]ering\b', 'differing'),
        (r'\bdi[`_ ]erently\b', 'differently'),
        (r'\bdi[`_ ]erent\b', 'different'),
        (r'\bdi[`_ ]ers\b', 'differs'),
        (r'\bdi[`_ ]er\b', 'differ'),
        (r'\bsu[`_ ]iciently\b', 'sufficiently'),
        (r'\bsu[`_ ]icient\b', 'sufficient'),
        (r'\bsu[`_ ]ered\b', 'suffered'),
        (r'\bsti[`_ ]ening\b', 'stiffening'),
        (r'\bsti[`_ ]ness\b', 'stiffness'),
        (r'\bcoe[`_ ]icient\b', 'coefficient'),
        (r'\bbu[`_ ]er\b', 'buffer'),
        (r'\bpara[`_ ]ins\b', 'paraffins'),
        (r'\bpara[`_ ]inic\b', 'paraffinic'),
        (r'\bo[`_^]set\b', 'offset'),
        (r'\bo_board\b', 'offboard'),
        (r'\bE[\^_`]lux\b', 'Efflux'),
        (r'\bInfux\b', 'Influx'),
        (r'\bfexible\b', 'flexible'),
        (r'\bftted\b', 'fitted'),
        (r'\bdefned\b', 'defined'),
        (r'\bfow\b', 'flow'),
        (r'\baailiates\b', 'affiliates'),
        (r'\baailiate\b', 'affiliate'),
        (r'\baaected\b', 'affected'),
        (r'\baaecting\b', 'affecting'),
        (r'\baaect\b', 'affect'),
        (r'\bagiliates\b', 'affiliates'),
        (r'\bagiliated\b', 'affiliated'),
        (r'\bagected\b', 'affected'),
        (r'\bagecting\b', 'affecting'),
        (r'\bagect\b', 'affect'),
        (r'\babecting\b', 'affecting'),
        (r'\bsta[`_](?!\w)', 'staff'),
        (r'\bstaa\b', 'staff'),
        (r'\bstag\b', 'staff'),
        (r'\bmidri[`_]\b', 'midriff'),
        (r'\bclassificatio\b', 'classification'),
        (r'\bthirdparty\b', 'third-party'),
        (r'\bca\s+rs\b', 'cars'),
    ]
    for pattern, replacement in patterns:
        text = _sub_word(text, pattern, replacement)

    # standalone/hyphenated off broken as o_, o`, o^, or og
    text = re.sub(r'\bo[`_\^](?!\w)', lambda m: _case_like(m.group(0), 'off'), text, flags=re.IGNORECASE)
    text = re.sub(r'\bog\b', lambda m: _case_like(m.group(0), 'off'), text, flags=re.IGNORECASE)

    # direct legacy replacements kept from original code
    broken_words = {
        r'\bSpeci ic\b': 'Specific',
        r'\bspeci ic\b': 'specific',
        r'\bSpeci ied\b': 'Specified',
        r'\bspeci ied\b': 'specified',
        r'\bspeci ication\b': 'specification',
        r'\bef iciently\b': 'efficiently',
        r'\bfundamental valued\b': 'fundamental values',
    }
    for broken, fixed in broken_words.items():
        text = re.sub(broken, fixed, text)

    # Strip leftover Markdown emphasis underscores that came from PDF italics.
    text = re.sub(r'(?<!\w)_([^_\n]+?)_(?!\w)', r'\1', text)
    text = re.sub(r'(?<=\s)_(?=\S)', '', text)
    text = re.sub(r'(?<=\S)_(?=\s|[.,;:)\]]|$)', '', text)
    text = re.sub(r'\bTRIM-\s+SURFACE\b', 'TRIM-SURFACE', text)
    text = re.sub(r'\bca\s+rs\b', 'cars', text)

    return text

def remove_noise(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # logo/picture blocks
    text = re.sub(
        r'\n\*\*==> picture \[\d+ x \d+\] intentionally omitted <==\*\*\n'
        r'(?:\n)?\*\*----- Start of picture text -----\*\*<br>\n'
        r'.+?<br>\*\*----- End of picture text -----\*\*<br>\n',
        '\n', text, flags=re.DOTALL
    )

    text = re.sub(r'(?m)^\s*2026 Formula 1:[^\n]+Fédération Internationale de l[’\']Automobile\s*$', '', text)
    text = re.sub(r'(?m)^\s*>?\s*\d{1,2}\s+[A-Za-z]+\s+\d{4}\s+[A-Z]\d+\s+Issue\s+\d+\s*$', '', text)
    text = re.sub(r'(?m)^\s*Version:\s*Issue\s+\d+[^\n]*$', '', text)
    text = re.sub(r'(?m)^\s*Issue\s+\d+\s*$', '', text)
    text = re.sub(r'(?m)^\s*(?!##\s*\*\*)SECTION\s+[A-Z]\s*:\s*[^\n]+$', '', text)
    text = re.sub(r'\n\|SECTION [A-Z]: [^\|]+\|[^\|]+\|\n\|---\|---\|\n', '\n', text)


    text = re.sub(r'(?m)^\s*-\s*$', '', text)
    text = re.sub(r'(?m)^\s*##\s*$', '', text)

    text = re.sub(r'## \*\*CONVENTION:\*\*.*?(?=## \*\*)', '', text, flags=re.DOTALL)
    text = re.sub(r'## \*\*CONTENTS:\*\*.*?(?=## \*\*(?:PREAMBLE|ARTICLE\s+[A-Z]))', '', text, flags=re.DOTALL)

    text = re.sub(r'(?m)^\s*_Advisory Committee:.*?_\s*$', '', text)
    text = re.sub(r'(?m)^\s*_Governance:.*?_\s*$', '', text)

    text = re.sub(r'[ \t]+\n', '\n', text)
    text = re.sub(r'(?<!\n) {2,}(?!\n)', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip() + '\n'

def clean_markdown(text: str) -> str:
    return remove_noise(fix_text_quality(text))

def quality_report(text: str) -> dict:
    return {
        'backticks': text.count('`'),
        'word_with_backtick': len(re.findall(r'\b\w*`\w*\b', text)),
        'word_with_underscore': len(re.findall(r'\b\w*_[\w_]*\b', text)),
        'empty_bullets': len(re.findall(r'(?m)^\s*-\s*$', text)),
        'double_spaces': len(re.findall(r'(?<!\n) {2,}(?!\n)', text)),
        'known_bad_tokens': sum(text.count(x) for x in ['o`icial', 'o_icial', 'e_ect', 'di_erent', 'aailiate', 'staa', 'ogicial', 'agect'])
    }

In [25]:
files = glob("../../data/rag/raw/fia_2026_f1_regulations_*.pdf")
print(f"발견된 PDF: {len(files)}개\n")

md_files = []

for pdf in files:
    print(f"변환 중: {pdf}")
    
    # pymupdf로 직접 변환 (pymupdf4llm 안 씀)
    doc = pymupdf.open(pdf)
    md = ""
    
    for page_num, page in enumerate(doc, start=1):
        # 페이지 텍스트 추출
        text = page.get_text("text")  # 또는 "blocks", "dict"
        md += f"\n## Page {page_num}\n\n{text}\n"
    
    doc.close()
    
    # 취소선 제거
    before_strikethrough = len(re.findall(r'~~.*?~~', md, flags=re.DOTALL))
    md = re.sub(r'~~.*?~~', '', md, flags=re.DOTALL)

    before_report = quality_report(md)
    md = clean_markdown(md)
    after_report = quality_report(md)

    md_file = pdf.replace('.pdf', '.md')
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(md)

    print(f"저장 완료: {md_file}")
    print(f"취소선 제거: {before_strikethrough}개")
    print(f"정리 전: {before_report}")
    print(f"정리 후: {after_report}\n")
    md_files.append(md_file)

print(f"{len(md_files)}개 파일 변환 완료!")

발견된 PDF: 5개

변환 중: ../../data/rag/raw\fia_2026_f1_regulations_-_section_a_general_provisions_-_iss_02_-_2026-02-27.pdf
저장 완료: ../../data/rag/raw\fia_2026_f1_regulations_-_section_a_general_provisions_-_iss_02_-_2026-02-27.md
취소선 제거: 0개
정리 전: {'backticks': 0, 'word_with_backtick': 0, 'word_with_underscore': 0, 'empty_bullets': 9, 'double_spaces': 120, 'known_bad_tokens': 14}
정리 후: {'backticks': 0, 'word_with_backtick': 0, 'word_with_underscore': 0, 'empty_bullets': 0, 'double_spaces': 0, 'known_bad_tokens': 0}

변환 중: ../../data/rag/raw\fia_2026_f1_regulations_-_section_b_sporting_-_iss_05_-_2026-02-27.pdf
저장 완료: ../../data/rag/raw\fia_2026_f1_regulations_-_section_b_sporting_-_iss_05_-_2026-02-27.md
취소선 제거: 0개
정리 전: {'backticks': 70, 'word_with_backtick': 66, 'word_with_underscore': 0, 'empty_bullets': 0, 'double_spaces': 156, 'known_bad_tokens': 32}
정리 후: {'backticks': 0, 'word_with_backtick': 0, 'word_with_underscore': 0, 'empty_bullets': 0, 'double_spaces': 0, 'known_bad_tokens': 0}
